[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mihiarc/socialmapper/blob/main/docs/notebooks/07-complete-analysis-workflow.ipynb)

# Complete Analysis Workflow — Food Desert Assessment

This notebook demonstrates a **complete equity analysis** using every core SocialMapper function. We will compare two Chicago neighborhoods — **Hyde Park** and **Pilsen** — to assess grocery access, income levels, and poverty rates.

A **food desert** is an area with limited access to affordable, nutritious food, often correlated with lower incomes and fewer vehicle-owning households.

### Workflow
1. Define two neighborhoods
2. Create walking isochrones (15 minutes)
3. Find grocery/food POIs in each area
4. Gather demographics (population, income, poverty, housing)
5. Map each neighborhood with POI overlays
6. Create income choropleth maps
7. Run formal multi-location comparison
8. Generate an HTML report
9. Summarize findings

## Setup

In [ ]:
# Uncomment to install on Google Colab:
# !pip install socialmapper

from socialmapper import (
    create_isochrone,
    get_census_blocks,
    get_census_data,
    create_map,
    get_poi,
    analyze_multiple_pois,
    generate_report,
)
from IPython.display import Image, display, HTML

## 1. Define Neighborhoods

We use central coordinates for each neighborhood.

In [ ]:
neighborhoods = {
    "Hyde Park": (41.7943, -87.5907),
    "Pilsen": (41.8525, -87.6514),
}

for name, coords in neighborhoods.items():
    print(f"{name}: lat={coords[0]}, lon={coords[1]}")

## 2. Walking Isochrones

For food-access analysis, walking isochrones are more relevant than driving — many residents in food deserts lack vehicle access.

In [ ]:
isochrones = {}
for name, coords in neighborhoods.items():
    iso = create_isochrone(coords, travel_time=15, travel_mode="walk")
    isochrones[name] = iso
    area = iso["properties"]["area_sq_km"]
    print(f"{name}: {area:.2f} sq km walkable in 15 min")

## 3. Find Grocery and Food POIs

Search for shopping (includes supermarket, grocery, convenience) within the walking isochrone.

In [ ]:
poi_data = {}
for name, coords in neighborhoods.items():
    pois = get_poi(
        coords,
        categories=["shopping", "food_and_drink"],
        travel_time=15,
        travel_mode="walk",
        limit=50,
    )
    poi_data[name] = pois
    print(f"\n{name}: {len(pois)} food/shopping POIs within 15-min walk")
    for p in pois[:5]:
        travel = p.get('travel_time_minutes', 'N/A')
        print(f"  {p['name']:<35} {p['category']:<20} {travel} min")

## 4. Gather Demographics

In [ ]:
demographic_variables = ["population", "median_income", "poverty", "housing_units", "households_no_vehicle"]

blocks_data = {}
census_data = {}
merged_data = {}

for name in neighborhoods:
    iso = isochrones[name]
    blocks = get_census_blocks(polygon=iso)
    census = get_census_data(iso, variables=demographic_variables)

    # Merge
    merged = []
    for block in blocks:
        geoid = block["geoid"]
        if geoid in census.data:
            merged.append({**block, **census.data[geoid]})

    blocks_data[name] = blocks
    census_data[name] = census
    merged_data[name] = merged

    print(f"\n{name}:")
    print(f"  Block groups: {len(merged)}")
    total_pop = sum(d.get("population", 0) for d in census.data.values() if d.get("population") is not None)
    print(f"  Total population: {total_pop:,}")

## 5. Population Maps with POI Overlays

In [ ]:
for name in neighborhoods:
    overlay_points = [
        {"lat": p["lat"], "lon": p["lon"], "name": p["name"]}
        for p in poi_data[name][:10]
    ]

    map_result = create_map(
        data=merged_data[name],
        column="population",
        title=f"Population — {name}, Chicago",
        overlay_boundary=isochrones[name],
        overlay_points=overlay_points,
        show_stats=True,
    )
    print(f"\n{name}:")
    display(Image(data=map_result.image_data))

## 6. Income Maps

In [ ]:
for name in neighborhoods:
    income_map = create_map(
        data=merged_data[name],
        column="median_income",
        title=f"Median Household Income — {name}",
        overlay_boundary=isochrones[name],
        show_stats=True,
        cmap="RdYlGn",
    )
    print(f"\n{name}:")
    display(Image(data=income_map.image_data))

## 7. Formal Multi-Location Comparison

Use `analyze_multiple_pois` for a structured side-by-side comparison.

In [ ]:
comparison = analyze_multiple_pois(
    locations=list(neighborhoods.values()),
    travel_time=15,
    travel_mode="walk",
    variables=["population", "median_income", "poverty", "housing_units"],
)

print("=== Comparison Rankings ===")
for var, info in comparison["comparison"].items():
    print(f"\n{var}:")
    for rank in info["ranked"]:
        print(f"  {rank['location']:<25} total={rank['total']:>10,.0f}  mean={rank['mean']:>8,.0f}")

## 8. Generate HTML Report

In [ ]:
report_html = generate_report(comparison, format="html")
print(f"Report: {len(report_html):,} characters")
display(HTML(report_html))

## 9. Key Findings

Summarize the analysis results.

In [ ]:
print("=" * 60)
print("FOOD ACCESS EQUITY ASSESSMENT")
print("Chicago: Hyde Park vs Pilsen (15-min walk)")
print("=" * 60)

for name in neighborhoods:
    census = census_data[name]
    pois = poi_data[name]
    iso = isochrones[name]

    total_pop = sum(
        d.get("population", 0)
        for d in census.data.values()
        if d.get("population") is not None
    )
    incomes = [
        d["median_income"]
        for d in census.data.values()
        if d.get("median_income") is not None and d["median_income"] > 0
    ]
    avg_income = sum(incomes) / len(incomes) if incomes else 0
    total_poverty = sum(
        d.get("poverty", 0)
        for d in census.data.values()
        if d.get("poverty") is not None
    )
    poverty_rate = (total_poverty / total_pop * 100) if total_pop > 0 else 0

    print(f"\n--- {name} ---")
    print(f"  Walkable area:        {iso['properties']['area_sq_km']:.2f} sq km")
    print(f"  Population:           {total_pop:,}")
    print(f"  Avg median income:    ${avg_income:,.0f}")
    print(f"  Poverty rate:         {poverty_rate:.1f}%")
    print(f"  Food/grocery POIs:    {len(pois)}")
    if total_pop > 0:
        print(f"  POIs per 1,000 people: {len(pois) / total_pop * 1000:.1f}")

## Next Steps

Ideas to extend this analysis:

- **More neighborhoods** — add additional areas to `analyze_multiple_pois`
- **Vehicle access** — compare `households_no_vehicle` across neighborhoods
- **Driving isochrones** — create drive-mode isochrones for areas with better car access
- **Time series** — compare census data across multiple years
- **Custom POI data** — use `import_poi_csv` to add your own food outlet data
- **Interactive maps** — use `export_format='html'` for zoomable, clickable maps

## API Cheat Sheet

| Function | Purpose |
|---|---|
| `create_isochrone(location, travel_time, travel_mode)` | Travel-time polygon |
| `get_census_blocks(polygon=iso)` | Block groups in an area |
| `get_census_data(location, variables)` | ACS demographic data |
| `create_map(data, column, ...)` | Choropleth visualization |
| `get_poi(location, categories, ...)` | OSM points of interest |
| `analyze_multiple_pois(locations, ...)` | Multi-location comparison |
| `generate_report(data, format)` | HTML/PDF report |
| `import_poi_csv(path)` | Custom POI data from CSV |